In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Agent & App Activity Analytics
# MAGIC Computes the metrics from the original proposal (alerts/hour, tool
# MAGIC success/error rate, most-watched routes, active users) from the
# MAGIC Lakebase Change Data Feed history tables (`lb_*_history`), and
# MAGIC writes them into a Delta table for a dashboard.
# MAGIC
# MAGIC Schema notes for these tables (confirmed via DESCRIBE):
# MAGIC   _pg_change_type  -> 'insert' | 'update' | 'delete' (Postgres-native, not Delta's own CDF vocabulary)
# MAGIC   _timestamp       -> when the change was captured
# MAGIC   action_id/user_id -> BINARY (raw Postgres UUID bytes) — fine for
# MAGIC                        grouping/counting, not human-readable as-is

# COMMAND ----------

dbutils.widgets.text("catalog", "bootcamp_students")
dbutils.widgets.text("schema", "madgula_sirisha_capstone")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

action_log_table = f"{catalog}.{schema}.lb_agent_action_log_history"
watched_flights_table = f"{catalog}.{schema}.lb_watched_flights_history"
alerts_table = f"{catalog}.{schema}.lb_alerts_history"

In [0]:
# MAGIC %md
# MAGIC ## 1. Agent tool success/error rate by action type
# MAGIC Filtered to inserts since agent_action_log is append-only in
# MAGIC Lakebase — every row is a fresh event, never updated.

# COMMAND ----------

tool_success_rate = spark.sql(f"""
    SELECT
        action_type,
        result_status,
        COUNT(*) AS event_count
    FROM {action_log_table}
    WHERE _pg_change_type = 'insert'
    GROUP BY action_type, result_status
    ORDER BY action_type, result_status
""")
display(tool_success_rate)

In [0]:
# MAGIC %md
# MAGIC ## 2. Tool call volume per hour (all tools, custom + MCP)

# COMMAND ----------

calls_per_hour = spark.sql(f"""
    SELECT
        date_trunc('hour', created_at) AS hour,
        COUNT(*) AS tool_calls
    FROM {action_log_table}
    WHERE _pg_change_type = 'insert'
    GROUP BY date_trunc('hour', created_at)
    ORDER BY hour
""")
display(calls_per_hour)

In [0]:
# MAGIC %md
# MAGIC ## 3. MCP vs. custom tool call split
# MAGIC Recall the logging middleware tags every MCP call as `mcp::<tool_name>`
# MAGIC — this splits activity by that prefix.

# COMMAND ----------

mcp_vs_custom = spark.sql(f"""
    SELECT
        CASE WHEN action_type LIKE 'mcp::%' THEN 'mcp' ELSE 'custom' END AS tool_source,
        COUNT(*) AS event_count
    FROM {action_log_table}
    WHERE _pg_change_type = 'insert'
    GROUP BY CASE WHEN action_type LIKE 'mcp::%' THEN 'mcp' ELSE 'custom' END
""")
display(mcp_vs_custom)

In [0]:

# MAGIC %md
# MAGIC ## 4. Alerts triggered per hour
# MAGIC Each update row in the history table reflects the row's new state at
# MAGIC that point in time, so filtering to status='triggered' on update rows
# MAGIC captures each transition into that state. An alert that fires,
# MAGIC un-fires, and re-fires would be double-counted — an acceptable
# MAGIC simplification given this project's scale.

alerts_per_hour = spark.sql(f"""
    SELECT
        date_trunc('hour', _timestamp) AS hour,
        COUNT(*) AS alerts_triggered
    FROM {alerts_table}
    WHERE _pg_change_type = 'update' AND status = 'triggered'
    GROUP BY date_trunc('hour', _timestamp)
    ORDER BY hour
""")
display(alerts_per_hour)

In [0]:
# MAGIC %md
# MAGIC ## 5. Most-watched routes

# COMMAND ----------

most_watched_routes = spark.sql(f"""
    SELECT
        origin_airport,
        destination_airport,
        COUNT(*) AS watch_count
    FROM {watched_flights_table}
    WHERE _pg_change_type = 'insert'
      AND origin_airport IS NOT NULL
      AND destination_airport IS NOT NULL
    GROUP BY origin_airport, destination_airport
    ORDER BY watch_count DESC
    LIMIT 20
""")
display(most_watched_routes)


In [0]:
# MAGIC %md
# MAGIC ## 6. Active users per day
# MAGIC user_id is BINARY, but COUNT(DISTINCT ...) works fine on raw bytes —
# MAGIC we don't need it human-readable to count distinct users.

# COMMAND ----------

active_users_per_day = spark.sql(f"""
    SELECT
        date_trunc('day', created_at) AS day,
        COUNT(DISTINCT user_id) AS active_users
    FROM {action_log_table}
    WHERE _pg_change_type = 'insert'
    GROUP BY date_trunc('day', created_at)
    ORDER BY day
""")
display(active_users_per_day)
